# Week 10 HW04: 입력 구조에 맞는 딥러닝 모델 선택

이 노트북은 이미지 분류에서는 MLP와 CNN을 비교하고, 텍스트 감성 분류에서는 LSTM과 Text CNN을 비교한다. 핵심 목적은 입력 데이터의 구조가 모델 설계에 어떤 영향을 주는지 확인하는 것이다.

## Part A. Image Task

- 데이터셋: MNIST
- 비교 모델: MLP baseline, CNN baseline
- 비교 항목: train / validation / test accuracy, loss 감소 그래프

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

cwd = Path.cwd()
if cwd.name == "notebooks":
    ROOT = cwd.parent
elif (cwd / "Assignment4").exists():
    ROOT = cwd / "Assignment4"
else:
    ROOT = Path("..")

with open(ROOT / "outputs" / "results" / "image_results.json", encoding="utf-8") as f:
    image_results = json.load(f)

image_rows = []
for model_name, metrics in image_results["models"].items():
    image_rows.append({
        "model": model_name,
        "train_accuracy": metrics["train_accuracy"],
        "validation_accuracy": metrics["validation_accuracy"],
        "test_accuracy": metrics["test_accuracy"],
    })
pd.DataFrame(image_rows)

In [ ]:
display(Image(filename=str(ROOT / "outputs" / "figures" / "image_loss_accuracy.png")))

### 이미지 모델 구조

MLP는 `28x28` 이미지를 `784`차원 벡터로 펼친 뒤 Dense layer로 분류한다. CNN은 `28x28x1`의 2차원 이미지 구조를 유지한 상태에서 Conv2D와 MaxPooling2D를 사용해 지역적인 시각 패턴을 학습한다.

## Part B. Text Task

- 데이터셋: NSMC 또는 수업 제공 한국어 리뷰 데이터
- 전처리: 한글/영문/숫자/공백만 남긴 뒤 Okt 가능 시 형태소 분석, 불가능하면 공백 토큰화
- 비교 모델: LSTM baseline, Text CNN baseline
- 비교 항목: validation / test accuracy, loss 감소 그래프

In [ ]:
with open(ROOT / "outputs" / "results" / "text_results.json", encoding="utf-8") as f:
    text_results = json.load(f)

text_rows = []
for model_name, splits in text_results["models"].items():
    text_rows.append({
        "model": model_name,
        "validation_accuracy": splits["validation"]["accuracy"],
        "test_accuracy": splits["test"]["accuracy"],
    })
pd.DataFrame(text_rows)

In [ ]:
display(Image(filename=str(ROOT / "outputs" / "figures" / "text_loss_accuracy.png")))

### 텍스트 모델 구조

LSTM은 Embedding 이후 양방향 LSTM을 사용해 단어 순서 정보를 반영한다. Text CNN은 Conv1D와 GlobalMaxPooling1D를 사용해 문장 안의 지역적인 n-gram 패턴을 포착한다.

## 해석

이미지는 픽셀의 2차원 위치 관계가 중요하므로 CNN이 MLP보다 유리하다. MLP는 이미지를 벡터로 펼치기 때문에 주변 픽셀 관계를 직접 보존하지 못하지만, CNN은 작은 필터를 이동시키며 획, 모서리, 부분 형태 같은 지역 패턴을 학습할 수 있다.

텍스트는 단어의 등장 여부뿐 아니라 순서가 의미에 영향을 준다. 따라서 LSTM 같은 sequence 모델은 앞뒤 단어 흐름을 반영할 수 있고, Text CNN은 짧은 구간의 표현 패턴을 빠르게 잡아낼 수 있다. 두 task를 비교하면 입력 구조의 차이가 모델 설계 방식에 직접적인 영향을 준다는 점을 확인할 수 있다.